# 머신러닝을 위한 선형대수학
## 01. 선형대수학 소개

---

### 목차
1. 머신러닝과 딥러닝에서의 선형대수학의 역할
2. 데이터 표현 (벡터 & 행렬)
3. 모델 표현
4. 행렬 연산
5. 차원 축소

In [ ]:
# ── 공통 라이브러리 import ───────────────────────────────────────────────
import numpy as np                          # 수치 배열·행렬 연산 (벡터, 행렬, 선형대수)
import matplotlib.pyplot as plt             # 그래프·시각화
import matplotlib.patches as mpatches       # 도형(박스, 화살표 등) 그리기
from matplotlib import rc                   # matplotlib 전역 설정

# 그래프 기본 스타일 설정 (노트북 전체에 적용)
plt.rcParams['font.family'] = 'DejaVu Sans'       # 기본 폰트 (수식·기호 표시)
plt.rcParams['figure.dpi'] = 100                  # 화면 해상도 (선명도)
plt.rcParams['axes.unicode_minus'] = False        # 마이너스 기호 깨짐 방지

print('라이브러리 로드 완료')


---

## 1. 데이터 표현 — 벡터(Vector)

선형대수는 주로 **행렬**과 **벡터**를 다루는 과목입니다.  
데이터는 행렬 또는 벡터의 형태로 나타낼 수 있으며, 선형대수는 이러한 데이터를 효율적으로 표현하고 조작하는 데 필수적입니다.

### 텍스트의 벡터 표현 (Bag of Words)

예를 들어, `"I love machine learning"` 이라는 문장은 4개의 단어로 구성됩니다.  
각 단어가 문장에 포함되면 1, 포함되지 않으면 0으로 표시하면:

$$
\text{vocab} = [\text{"I"},\ \text{"love"},\ \text{"machine"},\ \text{"learning"}]
$$

$$
\vec{v} = [1,\ 1,\ 1,\ 1] \in \mathbb{R}^4
$$

두 문장을 비교하면:

| 단어 | "I" | "love" | "machine" | "learning" |
|------|-----|--------|-----------|------------|
| 문장 1: "I love machine learning" | 1 | 1 | 1 | 1 |
| 문장 2: "I love deep learning"    | 1 | 1 | 0 | 1 |



### **기호별 세부 읽기**

* **$\vec{v}$** : 벡터 브이
* **$=$** : 은 / 는 (또는 같다)
* **$[1, 1, 1, 1]$** : 일, 일, 일, 일 (또는 대괄호 열고 일 콤마 일 콤마 일 콤마 일 대괄호 닫고)
* **$\in$** : ~에 속한다 / ~의 원소이다
* **$\mathbb{R}^4$** : 4차원 실수 공간 / 알 포 (R의 4제곱이라는 의미로 보통 '알 포'라고 많이 읽습니다.)

In [ ]:
# ── 텍스트의 벡터 표현 시각화 (Bag-of-Words) ─────────────────────────────
# NLP에서 문장을 고정 크기 숫자 벡터로 바꾸는 가장 단순한 방법: BoW
# 어휘(vocab) 각 단어가 1이면 등장, 0이면 미등장

vocab   = ['"I"', '"love"', '"machine"', '"learning"']  # 전체 어휘 목록 (4차원)
sent1   = np.array([1, 1, 1, 1])   # "I love machine learning" — 4단어 모두 등장
sent2   = np.array([1, 1, 0, 1])   # "I love deep learning" — 'machine'만 0 (미등장)

x       = np.arange(len(vocab))    # x축 위치: 0, 1, 2, 3
width   = 0.35                     # 막대 너비 (두 문장을 나란히 배치)

fig, ax = plt.subplots(figsize=(8, 4))

# 두 문장의 BoW 벡터를 나란히 막대그래프로 표시
bars1 = ax.bar(x - width/2, sent1, width, label='Sent 1: "I love machine learning"',
               color='#5C6BC0', alpha=0.85, edgecolor='white')
bars2 = ax.bar(x + width/2, sent2, width, label='Sent 2: "I love deep learning"',
               color='#EF5350', alpha=0.85, edgecolor='white')

ax.set_xticks(x)
ax.set_xticklabels(vocab, fontsize=12)
ax.set_yticks([0, 1])              # 이진 BoW이므로 0 또는 1만 표시
ax.set_ylabel('값 (0 or 1)', fontsize=11)
ax.set_title('Bag-of-Words 벡터 표현', fontsize=14, fontweight='bold', pad=12)
ax.legend(fontsize=10)
ax.set_ylim(0, 1.4)                # 막대 위 숫자 라벨 공간 확보
ax.spines[['top','right']].set_visible(False)  # 상·우 테두리 제거

# 각 막대 위에 실제 값(0 또는 1) 텍스트로 표시
for bar in bars1:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.04,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=11, color='#3949AB')
for bar in bars2:
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.04,
            str(int(bar.get_height())), ha='center', va='bottom', fontsize=11, color='#C62828')

plt.tight_layout()
plt.show()

print(f'문장 1 벡터: {sent1}')
print(f'문장 2 벡터: {sent2}')


---

## 2. 데이터 표현 — 행렬(Matrix)

이미지는 일반적으로 **픽셀 값의 행렬**로 표현됩니다.  
예를 들어, 45×40 크기의 RGB 이미지는 3개의 채널 행렬로 구성됩니다:

$$
\text{Image} \in \mathbb{R}^{45 \times 40 \times 3}
$$

각 채널의 픽셀 값 범위:

$$
x_{ij} \in \{0, 1, 2, \ldots, 255\}, \quad i \in [1, 45],\ j \in [1, 40]
$$

전체 이미지를 1차원 벡터로 펼치면 (flatten):

$$
\vec{x} = \text{flatten}(\text{Image}) \in \mathbb{R}^{45 \times 40 \times 3} = \mathbb{R}^{5400}
$$

In [ ]:
# ── 이미지의 행렬 표현 시각화 ─────────────────────────────────────────────
# 컬러 이미지 = (높이 × 너비 × 3) 3차원 텐서
# 각 채널(R, G, B)은 2차원 행렬(픽셀 밝기 0~255)

np.random.seed(42)  # 재현 가능한 랜덤 이미지
img_rgb   = np.random.randint(80, 256, (45, 40, 3), dtype=np.uint8)  # 45×40 RGB

# 그레이스케일 변환: ITU-R BT.601 가중합 (R:0.299, G:0.587, B:0.114)
gray      = (0.299*img_rgb[:,:,0] + 0.587*img_rgb[:,:,1] + 0.114*img_rgb[:,:,2]).astype(np.uint8)

fig, axes = plt.subplots(1, 4, figsize=(14, 4))
ch_colors = ['Reds', 'Greens', 'Blues']
ch_labels = ['R channel', 'G channel', 'B channel']

# ① 원본 RGB 합성 이미지
axes[0].imshow(img_rgb)
axes[0].set_title('RGB Image\n(45×40×3)', fontsize=11, fontweight='bold')
axes[0].axis('off')

# ②~④ R, G, B 채널 각각 단독 행렬로 분리해 표시
for i in range(3):
    ch_img = np.zeros((45, 40, 3), dtype=np.uint8)  # RGB 3채널 빈 배열
    ch_img[:, :, i] = img_rgb[:, :, i]              # i번째 채널만 채움
    axes[i+1].imshow(ch_img)
    axes[i+1].set_title(f'{ch_labels[i]}\n(45×40 matrix)', fontsize=11, fontweight='bold')
    axes[i+1].axis('off')

fig.suptitle('이미지 = 픽셀 값의 행렬 (RGB 3채널 분해)', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

# 행렬로 저장된 픽셀 값 일부 출력
print('R 채널 (처음 5×5 픽셀):')
print(img_rgb[:5, :5, 0])
print(f'\n이미지 shape: {img_rgb.shape}')           # (45, 40, 3)
print(f'Flatten 벡터 크기: {img_rgb.flatten().shape[0]}')  # 45×40×3 = 5400 (1D 벡터)


---

## 3. 모델 표현 — 선형 회귀 & 신경망

많은 머신러닝 모델이 **선형대수적 표현**을 사용합니다.  
신경망의 한 레이어는 다음과 같이 표현됩니다:

$$
u_k = w_{k1}x_1 + w_{k2}x_2 + \cdots + w_{kn}x_n + b_k
$$

이를 행렬 형태로 묶으면:

$$
\mathbf{u} = W\mathbf{x} + \mathbf{b}
$$

$$
\begin{pmatrix} u_1 \\ u_2 \\ \vdots \\ u_k \end{pmatrix}
=
\begin{pmatrix} w_{11} & w_{12} & \cdots & w_{1n} \\ w_{21} & w_{22} & \cdots & w_{2n} \\ \vdots & & \ddots & \vdots \\ w_{k1} & w_{k2} & \cdots & w_{kn} \end{pmatrix}
\begin{pmatrix} x_1 \\ x_2 \\ \vdots \\ x_n \end{pmatrix}
+
\begin{pmatrix} b_1 \\ b_2 \\ \vdots \\ b_k \end{pmatrix}
\Rightarrow
\begin{pmatrix} f(u_1) \\ f(u_2) \\ \vdots \\ f(u_k) \end{pmatrix}
$$

여기서 $f$는 **활성화 함수(activation function)** 입니다.

In [ ]:
# ── 선형 레이어 계산 & 활성화 함수 시각화 ────────────────────────────────
# 신경망 1층: u = Wx + b (행렬-벡터 곱 + 편향), y = f(u) (비선형 활성화)

np.random.seed(0)
n_in, n_out = 3, 4   # 입력 3차원 → 출력 4차원 (은닉츠 4개)

x_input = np.array([[1.0], [0.5], [-0.3]])          # 입력 열벡터 (3×1)
W       = np.random.randn(n_out, n_in).round(2)     # 가중치 행렬 (4×3): 각 출력 뉴런의 가중치
b       = np.random.randn(n_out, 1).round(2)         # 편향 열벡터 (4×1)
u       = W @ x_input + b                            # 선형 결합 (4×1): Wx + b
y_relu  = np.maximum(0, u)                           # ReLU: 음수는 0, 양수는 그대로

# 수치 결과 출력
print('=== 선형 레이어 계산 ===')
print(f'입력 x (3×1):\n{x_input}')
print(f'\n가중치 W (4×3):\n{W}')
print(f'\n편향 b (4×1):\n{b}')
print(f'\n선형 결합 u = Wx + b (4×1):\n{u.round(3)}')
print(f'\nReLU(u) = max(0, u) (4×1):\n{y_relu.round(3)}')

# ── 주요 활성화 함수 3종 시각화 ──────────────────────────────────────────
z = np.linspace(-3, 3, 300)   # 입력 z 범위
relu    = np.maximum(0, z)              # ReLU: max(0, z)
sigmoid = 1 / (1 + np.exp(-z))          # Sigmoid: (0, 1) 확률 해석
tanh    = np.tanh(z)                    # Tanh: (-1, 1) 대칭

fig, axes = plt.subplots(1, 3, figsize=(13, 4))
specs = [
    (relu,    '#5C6BC0', 'ReLU',    r'$f(z) = \max(0, z)$'),
    (sigmoid, '#EF5350', 'Sigmoid', r'$f(z) = \frac{1}{1+e^{-z}}$'),
    (tanh,    '#26A69A', 'Tanh',    r'$f(z) = \tanh(z)$'),
]
for ax, (y_vals, color, name, formula) in zip(axes, specs):
    ax.plot(z, y_vals, color=color, linewidth=2.5)
    ax.axhline(0, color='gray', linewidth=0.7, linestyle='--')  # x축
    ax.axvline(0, color='gray', linewidth=0.7, linestyle='--')  # y축
    ax.set_title(f'{name}\n{formula}', fontsize=11, fontweight='bold')
    ax.set_xlabel('z', fontsize=10)
    ax.set_ylabel('f(z)', fontsize=10)
    ax.spines[['top','right']].set_visible(False)
    ax.grid(True, alpha=0.3)

fig.suptitle('주요 활성화 함수 (Activation Functions)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---

## 4. 행렬 연산 (Matrix Operations)

많은 머신러닝 알고리즘이 **행렬 연산**을 기반으로 합니다.

### 주요 연산

| 연산 | 표기 | 설명 |
|------|------|------|
| 전치(Transpose) | $A^T$ | 행과 열을 교환 |
| 행렬 곱(Matrix Multiply) | $AB$ | $(AB)_{ij} = \sum_k A_{ik}B_{kj}$ |
| 역행렬(Inverse) | $A^{-1}$ | $AA^{-1} = I$ |
| 내적(Dot Product) | $\mathbf{a}^T\mathbf{b}$ | $\sum_i a_i b_i$ |

### 전치 행렬

$$
A = \begin{pmatrix} 1 & 2 & 3 \\ 4 & 5 & 6 \end{pmatrix}_{2\times3}
\quad\Rightarrow\quad
A^T = \begin{pmatrix} 1 & 4 \\ 2 & 5 \\ 3 & 6 \end{pmatrix}_{3\times2}
$$

### 역행렬과 선형 방정식

$$
A\mathbf{x} = \mathbf{b} \quad\Rightarrow\quad \mathbf{x} = A^{-1}\mathbf{b}
$$

In [ ]:
# ── 행렬 연산 계산 & 선형 변환 시각화 ────────────────────────────────────
# 전치(A^T), 행렬곱(AB), 역행렬(A^{-1}), 행렬=기하학적 선형 변환

A = np.array([[1, 2, 3],
              [4, 5, 6]])   # 2×3 행렬
B = np.array([[1, 2],
              [3, 4],
              [5, 6]])       # 3×2 행렬 → AB는 (2×3)(3×2) = 2×2

print('=== 행렬 연산 ===')
print(f'A (2×3):\n{A}')
print(f'\nA^T (3×2):\n{A.T}')      # 행↔열 교환
print(f'\nB (3×2):\n{B}')
print(f'\nAB (2×2):\n{A @ B}')     # @ : 행렬 곱셈 연산자

# ── 2×2 역행렬 예시 ─────────────────────────────────────────────────────
M = np.array([[2., 1.], [5., 3.]])
M_inv = np.linalg.inv(M)              # A @ A^{-1} = I (단위행렬)
print(f'\nM:\n{M}')
print(f'M^(-1):\n{M_inv.round(3)}')
print(f'M @ M^(-1) ≈ I:\n{(M @ M_inv).round(6)}')

# ── 행렬 = 선형 변환: 단위 원 → 타원(또는 다른 도형) ─────────────────────
theta = np.linspace(0, 2*np.pi, 300)           # 원 위 300개 점
circle = np.array([np.cos(theta), np.sin(theta)])  # 단위원 (반지름 1)

transform_M = np.array([[2., 1.], [0.5, 1.5]])   # 2×2 변환 행렬
transformed = transform_M @ circle                # 각 점에 M 적용 → 원이 찌그러짐

fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))

# ① 변환 전: 단위 원 + 기저 벡터 e1(1,0), e2(0,1)
axes[0].plot(circle[0], circle[1], '#5C6BC0', linewidth=2)
axes[0].quiver([0,0],[0,0],[1,0],[0,1],
               angles='xy', scale_units='xy', scale=1,
               color=['#E53935','#43A047'], width=0.012)
axes[0].set_xlim(-2, 2); axes[0].set_ylim(-2, 2)
axes[0].set_aspect('equal'); axes[0].grid(True, alpha=0.3)
axes[0].axhline(0, color='k', linewidth=0.5)
axes[0].axvline(0, color='k', linewidth=0.5)
axes[0].set_title('변환 전 (단위 원)', fontsize=12, fontweight='bold')
axes[0].spines[['top','right']].set_visible(False)

# ② 변환 후: M이 원·기저벡터를 어떻게 바꾸는지 표시
axes[1].plot(transformed[0], transformed[1], '#EF5350', linewidth=2)
axes[1].quiver([0,0],[0,0],
               transform_M[:,0], transform_M[:,1],   # M의 열벡터 = 변환된 기저
               angles='xy', scale_units='xy', scale=1,
               color=['#E53935','#43A047'], width=0.012)
axes[1].set_xlim(-4, 4); axes[1].set_ylim(-3, 3)
axes[1].set_aspect('equal'); axes[1].grid(True, alpha=0.3)
axes[1].axhline(0, color='k', linewidth=0.5)
axes[1].axvline(0, color='k', linewidth=0.5)
lbl = f'M = [[{transform_M[0,0]},{transform_M[0,1]}],[{transform_M[1,0]},{transform_M[1,1]}]]'
axes[1].set_title(f'변환 후\n{lbl}', fontsize=12, fontweight='bold')
axes[1].spines[['top','right']].set_visible(False)

fig.suptitle('행렬 = 선형 변환 (단위 원의 변환)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()


---

## 5. 차원 축소 (Dimensionality Reduction)

선형대수는 **주성분 분석(PCA)** 및 **특이값 분해(SVD)** 와 같은 차원 축소 기술에 필수적입니다.

### PCA의 핵심 수식

데이터 행렬 $X \in \mathbb{R}^{n \times d}$ 에서 공분산 행렬을 계산:

$$
\Sigma = \frac{1}{n-1} X^T X \in \mathbb{R}^{d \times d}
$$

고유값 분해(Eigendecomposition):

$$
\Sigma = V \Lambda V^T, \quad \Lambda = \text{diag}(\lambda_1, \lambda_2, \ldots, \lambda_d), \quad \lambda_1 \geq \lambda_2 \geq \cdots \geq \lambda_d
$$

상위 $k$개의 주성분으로 차원 축소:

$$
X_{\text{reduced}} = X V_k, \quad V_k \in \mathbb{R}^{d \times k}
$$

분산 설명 비율(Explained Variance Ratio):

$$
\text{EVR}_k = \frac{\sum_{i=1}^{k} \lambda_i}{\sum_{i=1}^{d} \lambda_i}
$$

In [ ]:
# ── PCA 차원 축소 시각화 ──────────────────────────────────────────────────
# PCA: 고차원 데이터를 분산이 큰 방향(주성분)으로 투영해 차원 축소

from sklearn.decomposition import PCA
from sklearn.datasets import make_classification
from sklearn.preprocessing import StandardScaler

np.random.seed(42)

# 3차원 분류 데이터 200개 생성 (2개 클래스)
X_3d, y = make_classification(n_samples=200, n_features=3, n_informative=2,
                               n_redundant=1, n_clusters_per_class=1, random_state=42)
X_scaled = StandardScaler().fit_transform(X_3d)  # 평균 0, 분산 1로 표준화 (PCA 전 필수)

# 3개 주성분 전부 추출
pca_full = PCA(n_components=3)
X_pca    = pca_full.fit_transform(X_scaled)       # PCA 좌표계로 변환
evr      = pca_full.explained_variance_ratio_     # 각 PC가 설명하는 분산 비율
cumulative_evr = np.cumsum(evr)                 # 누적 분산 설명률

print('=== PCA 결과 ===')
for i, (e, c) in enumerate(zip(evr, cumulative_evr)):
    print(f'PC{i+1}: 분산 설명률 = {e:.3f} ({e*100:.1f}%), 누적 = {c*100:.1f}%')

# ── 3-panel 시각화 ───────────────────────────────────────────────────────
colors = np.where(y == 0, '#5C6BC0', '#EF5350')
fig = plt.figure(figsize=(15, 4.5))

# ① 원본 3D 산점도
ax1 = fig.add_subplot(131, projection='3d')
ax1.scatter(X_scaled[y==0,0], X_scaled[y==0,1], X_scaled[y==0,2],
            c='#5C6BC0', alpha=0.6, s=20, label='Class 0')
ax1.scatter(X_scaled[y==1,0], X_scaled[y==1,1], X_scaled[y==1,2],
            c='#EF5350', alpha=0.6, s=20, label='Class 1')
ax1.set_title('원본 3D 데이터', fontsize=11, fontweight='bold')
ax1.set_xlabel('X1'); ax1.set_ylabel('X2'); ax1.set_zlabel('X3')
ax1.legend(fontsize=9)

# ② PC1-PC2 평면에 2D 투영 (3D → 2D)
ax2 = fig.add_subplot(132)
ax2.scatter(X_pca[y==0,0], X_pca[y==0,1], c='#5C6BC0', alpha=0.6, s=20, label='Class 0')
ax2.scatter(X_pca[y==1,0], X_pca[y==1,1], c='#EF5350', alpha=0.6, s=20, label='Class 1')
ax2.set_xlabel(f'PC1 ({evr[0]*100:.1f}%)', fontsize=10)
ax2.set_ylabel(f'PC2 ({evr[1]*100:.1f}%)', fontsize=10)
ax2.set_title('PCA: 3D → 2D', fontsize=11, fontweight='bold')
ax2.spines[['top','right']].set_visible(False)
ax2.legend(fontsize=9)
ax2.grid(True, alpha=0.3)

# ③ 각 PC별 분산 설명률 + 누적 곡선
ax3 = fig.add_subplot(133)
bars = ax3.bar(['PC1', 'PC2', 'PC3'], evr * 100,
               color=['#5C6BC0','#7E57C2','#AB47BC'],
               alpha=0.85, edgecolor='white')
ax3.plot(['PC1','PC2','PC3'], cumulative_evr * 100,
         'o--', color='#EF5350', linewidth=2, markersize=7, label='누적 분산 설명률')
ax3.set_ylabel('분산 설명률 (%)', fontsize=10)
ax3.set_title('Explained Variance Ratio', fontsize=11, fontweight='bold')
ax3.set_ylim(0, 115)
ax3.legend(fontsize=9)
ax3.spines[['top','right']].set_visible(False)
ax3.grid(True, alpha=0.3, axis='y')

for bar, val in zip(bars, evr * 100):
    ax3.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1.5,
             f'{val:.1f}%', ha='center', va='bottom', fontsize=10, fontweight='bold')

plt.suptitle('PCA를 이용한 차원 축소', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()


---

## 6. 강의 커리큘럼 요약

| 파트 | 이론 | 주요 내용 |
|------|------|-----------|
| **1. 기초** | 이론 2 | 벡터와 벡터 연산 (덧셈, 뺄셈, 내적, 크기) |
| **1. 기초** | 이론 3 | 행렬의 정의와 연산 (덧셈, 곱셈, 전치, 역행렬) |
| **2. 심화** | 이론 4 | 벡터 공간, 부분 공간, 기저, 차원, 직교기저 |
| **2. 심화** | 이론 5 | 선형 변환 (행렬 표현, 행렬식, 역행렬, 직교행렬) |
| **2. 심화** | 이론 6 | 고유값·고유벡터, 행렬 대각화, 스펙트럼 클러스터링 |
| **2. 심화** | 이론 7 | 특이값 분해(SVD), 영상 압축 응용 |
| **3. ML 응용** | 이론 8 | PCA를 이용한 특징 추출, 아이겐 페이스 |
| **3. ML 응용** | 이론 9 | 확률론 기초 (나이브 베이즈, 마르코프 결정 과정) |
| **3. ML 응용** | 이론 10-11 | 미분, 편미분, 경사 하강법, 라그랑주 승수법 |
| **4. DL 응용** | 이론 12 | 신경망 구조, 역전파, Gradient Descent 실습 |

---

> 💡 **공부 방법**: 너무 처음부터 다 이해하려 하지 않아도 됩니다.  
> 각 이론마다 구체적인 예를 들어 설명하고, 가능하면 **손으로 따라 쓰면서** 학습하세요.

In [ ]:
# ── 커리큘럼 로드맵 시각화 ────────────────────────────────────────────────
# 선형대수 학습 경로: 기초 → 심화 → ML → DL

fig, ax = plt.subplots(figsize=(12, 5))
ax.set_xlim(0, 12); ax.set_ylim(0, 5); ax.axis('off')  # 좌표만 쓰는 캔버스

# (x중심, y중심, 제목, 색상, 부제)
modules = [
    (0.5,  2.5, '1. 선형대수\n   기초',     '#5C6BC0', '이론 2-3\n벡터·행렬'),
    (3.0,  2.5, '2. 선형대수\n   심화',     '#8E24AA', '이론 4-7\n공간·SVD'),
    (5.5,  2.5, '3. ML 응용',             '#00897B', '이론 8-11\nPCA·미분'),
    (8.0,  2.5, '4. DL 응용',             '#D81B60', '이론 12\n신경망'),
    (10.5, 2.5, '머신러닝\n마스터!',        '#F57F17', ''),
]

for (x, y, title, color, sub) in modules:
    # 둥근 모서리 박스로 각 학습 단계 표시
    box = mpatches.FancyBboxPatch((x-1.0, y-0.85), 2.0, 1.7,
                                   boxstyle='round,pad=0.1',
                                   facecolor=color, edgecolor='white',
                                   linewidth=2, alpha=0.9)
    ax.add_patch(box)
    ax.text(x, y+0.25, title, ha='center', va='center',
            fontsize=10, fontweight='bold', color='white')
    if sub:
        ax.text(x, y-0.45, sub, ha='center', va='center',
                fontsize=8.5, color='white', alpha=0.9)

# 모듈 사이 화살표로 학습 순서 표시
arrow_xs = [1.5, 4.0, 6.5, 9.0]
for ax_x in arrow_xs:
    ax.annotate('', xy=(ax_x+0.5, 2.5), xytext=(ax_x, 2.5),
                arrowprops=dict(arrowstyle='->', color='#555', lw=2))

ax.set_title('강의 커리큘럼 로드맵', fontsize=14, fontweight='bold', pad=15)
plt.tight_layout()
plt.show()
